In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
import importlib
import yaml
import torch

## Test PL module

In [4]:
import lightning_scripts.lightning_ssl_sep_classifier_opt as lightning 
# import lightning_scripts.lightning_ssl as lightning 
importlib.reload(lightning)

LitAudioSSL = lightning.LitAudioSSL
## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/pilot_ssl_barlow_dualtask_resnet50_hparam_set_1_lr_06_LARS_sep_classifier_opt_no_lr_schedule.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 4 
config['num_gpus'] = 1 
config['hparas']['global_batch_size'] = 72
config['hparas']['batch_size'] =  config['hparas']['global_batch_size'] // config['num_gpus']  # total batch size / n gpus 

In [5]:
from pytorch_lightning.loggers import WandbLogger


In [6]:
config['hparas']

{'epochs': 10,
 'global_batch_size': 72,
 'lambda_ssl': 1,
 'lr': 0.6,
 'lr_schedule': False,
 'num_warmup_steps_or_ratio': 0.1,
 'optimizer': 'LARS',
 'ssl_task': 'dual',
 'ssl_loss': 'Paired_Loss',
 'ssl_loss_str': 'paired',
 'ssl_loss_kwargs': {'loss_fn_inv': 'Barlow_Loss',
  'loss_fn_eq': 'Barlow_Loss',
  'loss_fn_inv_kwargs': {'lmbda': 0.005, 'out_dim': 512},
  'loss_fn_eq_kwargs': {'lmbda': 0.005, 'out_dim': 512},
  'lmda': 0.5},
 'valid_step': 5000,
 'sep_class_opt': True,
 'batch_size': 72}

In [7]:
model = LitAudioSSL(config)

In [8]:
## Sanity check learning rate updates 

# # logic for total training steps:
# dataset_size = len(model.train_dataloader()) # should be pre-batched 
# print(dataset_size)
# num_devices = 4 # eg 1 
# accum_grad_batches = 1 # always 1 here unless simulating larger batch size 
# effective_batch_size = accum_grad_batches * num_devices
# max_estimated_steps = (dataset_size // effective_batch_size) * config['hparas']['epochs']

In [9]:
from lightning.pytorch.callbacks import ModelCheckpoint

from pathlib import Path

In [10]:
callbacks = []
lr_monitor = LearningRateMonitor(logging_interval='step')
callbacks.append(lr_monitor)
checkpoint_dir = Path('exp') / Path(config_path).stem / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# if config.get('val_metric', None):
#     callbacks.append(ModelCheckpoint(
#                             checkpoint_dir,
#                             monitor=f"{config['val_metric']}",
#                             mode=config['val_metric_mode'],
#                             save_top_k=1,
#                             save_weights_only=True,
#                             verbose=True,
#                 ))
# callbacks.append(ModelCheckpoint(
#             checkpoint_dir,
#             monitor="train_total_loss",
#             mode="min",
#             save_top_k=1,
#             save_weights_only=True,
#             verbose=True,
#         ))

In [11]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=10,
                    callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    gradient_clip_val=False,
                    devices=1)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [12]:
trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /mnt/ceph/users/igriffith/projects/cochdnn/lightning_logs/version_4148234/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                       | Params | Mode 
------------------------------------------------------------------
0 | transforms | AudioCompose               | 0      | train
1 | audio_rep  | AudioToAudioRepresentation | 0      | train
2 | model      | ModelWithFrontEnd          | 26.4 M | train
3 | ssl_loss   | Paired_Loss                | 0      | train
4 | class_loss | CrossEntropyLoss           | 0      | train
------------------------------------------------------------------
26.4 M    Trainable params
0         Non-trainable params
26.4 M    Total params
105.762   Total estimated model params size (MB)
174       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998


...

current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998


Validation: |          | 0/? [00:00<?, ?it/s]

current lr:  0.16874999999999998
current lr:  0.16874999999999998
current lr:  0.16874999999999998



Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [10]:
model.total_training_steps()

402000

In [19]:
model.trainer.lr_scheduler_configs[0].scheduler.optimizer.param_groups[0]['lr']

6.296641791044775e-05

In [22]:
model.optimizer.param_groups[0]

{'params': [Parameter containing:
  tensor([[[[ 0.0094,  0.0206,  0.0030,  ...,  0.0238,  0.0366,  0.0372],
            [-0.0154, -0.0214, -0.0246,  ...,  0.0180, -0.0114,  0.0198],
            [-0.0127,  0.0610, -0.0481,  ...,  0.0159,  0.0233, -0.0060],
            ...,
            [-0.0329, -0.0143,  0.0053,  ...,  0.0133,  0.0184,  0.0091],
            [-0.0292,  0.0647,  0.0096,  ...,  0.0222, -0.0316,  0.0290],
            [ 0.0225, -0.0234,  0.0090,  ...,  0.0124, -0.0528,  0.0173]]],
  
  
          [[[ 0.0138, -0.0037, -0.0008,  ...,  0.0231, -0.0282, -0.0189],
            [ 0.0043, -0.0211,  0.0212,  ..., -0.0458, -0.0427, -0.0337],
            [ 0.0401,  0.0292,  0.0024,  ...,  0.0132,  0.0063,  0.0063],
            ...,
            [-0.0335,  0.0304, -0.0425,  ...,  0.0031,  0.0065, -0.0131],
            [ 0.0054,  0.0116, -0.0147,  ...,  0.0044, -0.0016,  0.0103],
            [-0.0114, -0.0244,  0.0128,  ...,  0.0024,  0.0184, -0.0015]]],
  
  
          [[[-0.0165, -0.002

In [16]:
# init_lr = 0.6 * 768 / 256
warmup = model.compute_warmup(model.total_training_steps(), 0.1)

In [17]:
warmup

40200.0

In [10]:
model.total_training_steps()

-1

In [12]:
dataset_size = len(model.train_dataloader())
num_devices = model.config['num_gpus']
effective_batch_size = model.trainer.accumulate_grad_batches * num_devices
max_estimated_steps = (dataset_size // effective_batch_size) * model.trainer.max_epochs

if model.trainer.max_steps and model.trainer.max_steps < max_estimated_steps:
    print(int(model.trainer.max_steps))
int(max_estimated_steps)

-1


452000

In [18]:
%reload_ext tensorboard


ModuleNotFoundError: No module named 'tensorboard'